# AMPS II · Week 14 — HW13
## Growth rates, the interface, Flory–Huggins, solvers and scaling, a reused factorisation and an LLM-generated phase-field script

**Due before Week 15 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q7 runs an LLM-generated script that is **meant** to be wrong. The whole HW runs in about 7 minutes locally, about 25 on Colab. Timings: compare ratios.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Theory: linear stability, amplification factors, convex splitting, the common tangent | 15 |
| 2 | Growth rates: time and space convergence of three schemes | 14 |
| 3 | The equilibrium interface: profile and interfacial energy | 14 |
| 4 | Flory–Huggins: binodal compositions and a failing Newton step | 13 |
| 5 | Linear solvers: cost, iterations, memory, ghosts | 15 |
| 6 | Accuracy per cost: Newton against a reused factorisation | 14 |
| 7 | Debugging an LLM-generated Cahn–Hilliard script | 15 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model module from the lab (unchanged). Run the cell once.

In [ ]:
%%writefile /tmp/fnx/phase.py
"""phase.py — Cahn-Hilliard phase separation with energy-stable time stepping, and parallel-performance probes, DOLFINx 0.11 (AMPS II, Week 14)."""
import gc, time
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import NonlinearProblem
import dolfinx.fem.petsc as fp
import basix.ufl

comm = MPI.COMM_WORLD
gc.disable()                                   # PETSc destructors are collective: collect at the same point on all ranks
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-12, snes_stol=0.0, snes_max_it=25)

def box(L=1.0, H=1.0, nx=32, ny=32, x0=0.0):
    return mesh.create_rectangle(comm, [[x0, 0], [x0 + L, H]], [nx, ny], mesh.CellType.quadrilateral)

def noise(x, amp=0.05, seed=1, modes=40):
    """a smooth random field as a function of the coordinates (the same on every partition, unlike np.random per dof)"""
    r = np.random.default_rng(seed); s = np.zeros(x.shape[1])
    for _ in range(modes):
        kx, ky = r.integers(1, 12, 2); ph = r.uniform(0, 2 * np.pi, 2)
        s += np.cos(kx * np.pi * x[0] + ph[0]) * np.cos(ky * np.pi * x[1] + ph[1])
    return amp * s / 10

def line(msh, f, y=0.5, x0=0.0, x1=1.0, n=401):
    """x and f(x, y) along a horizontal line (every rank gets all values)"""
    P = np.zeros((n, 3)); P[:, 0] = np.linspace(x0, x1, n); P[:, 1] = y
    tdim = msh.topology.dim; nloc = msh.topology.index_map(tdim).size_local
    cand = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, tdim), P), P)
    out, cnt = np.zeros(n), np.zeros(n); idx, cells = [], []
    for j in range(n):
        own = [c for c in cand.links(j) if c < nloc]
        if own:
            idx.append(j); cells.append(own[0])
    if idx:
        out[idx] = np.asarray(f.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)[:, 0]; cnt[idx] = 1
    return P[:, 0], comm.allreduce(out, op=MPI.SUM) / np.maximum(comm.allreduce(cnt, op=MPI.SUM), 1)

# ═══════════════════════════════════════════════ free energies
def quartic(c):
    """f(c) = (c^2 - 1)^2 / 4 (phases c = -1, +1)"""
    return 0.25 * (c ** 2 - 1) ** 2

def flory_huggins(phi, chi, N1=1.0, N2=1.0):
    """f(phi) = phi/N1 ln phi + (1 - phi)/N2 ln(1 - phi) + chi phi (1 - phi) (lattice units)"""
    return phi / N1 * ufl.ln(phi) + (1 - phi) / N2 * ufl.ln(1 - phi) + chi * phi * (1 - phi)

class CahnHilliard:
    """c_t = div(M grad mu), mu = f'(c) - eps^2 lap c on a box with no-flux walls; mixed P1 x P1 (c, mu).
    scheme: "be" (fully implicit backward Euler, Newton), "cs" (convex splitting, Eyre: implicit convex part, explicit concave part),
            "stab" (linear, stabilised: f'(c_n) + S (c - c_n); one constant matrix), "cn" (mu averaged over the step: not energy stable).
    potential: "quartic" (f = (c^2-1)^2/4) or "fh" (Flory-Huggins with chi; c is the volume fraction phi in (0, 1))."""
    count = 0
    def __init__(self, msh, eps=0.05, M=1.0, dt=1e-4, scheme="cs", S=2.0, potential="quartic", chi=3.0, deg=1, solver=None):
        collect(); CahnHilliard.count += 1; k = CahnHilliard.count
        self.msh, self.scheme, self.potential, self.chi = msh, scheme, potential, chi
        cell = msh.basix_cell(); P = basix.ufl.element("Lagrange", cell, deg)
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([P, P]))
        self.w, self.w0 = fem.Function(W), fem.Function(W)
        c, mu = ufl.split(self.w); c0, mu0 = ufl.split(self.w0); q, v = ufl.TestFunctions(W)
        self.dt = fem.Constant(msh, float(dt)); self.eps = eps; self.M = M
        if potential == "quartic":
            fc, fe = (lambda c: c ** 3), (lambda c: c)                  # f' = c^3 - c: convex part c^3, concave part -c
            fp_ = lambda c: c ** 3 - c
        else:                                                           # Flory-Huggins: convex part ln terms, concave part chi (1 - 2 phi)
            fc = lambda c: ufl.ln(c / (1 - c)) + 0 * c
            fe = lambda c: -chi * (1 - 2 * c)
            fp_ = lambda c: ufl.ln(c / (1 - c)) + chi * (1 - 2 * c)
        mob = mu                                                        # the chemical potential in the mobility term
        if scheme == "be":
            dfdc = fp_(c)
        elif scheme == "cn":                                            # theta = 1/2 on mu (as often written; NOT energy stable for Cahn-Hilliard)
            dfdc = fp_(c); mob = 0.5 * (mu + mu0)
        elif scheme == "cs":
            dfdc = fc(c) - fe(c0)
        elif scheme == "stab":
            dfdc = fp_(c0) + S * (c - c0)
        else:
            raise ValueError(scheme)
        F = ((c - c0) / self.dt * q + M * ufl.dot(ufl.grad(mob), ufl.grad(q)) + (mu - dfdc) * v - eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(v))) * ufl.dx
        self.F = F
        opts = dict(NEWTON) if solver is None else dict(NEWTON, **solver)
        self.pb = NonlinearProblem(F, self.w, petsc_options_prefix=f"ch{k}_", petsc_options=opts)
        self.Vc, self.mapc = W.sub(0).collapse()
        self.t = 0.0; self.its = []

    def set_c(self, fun):
        """initial composition c(x) (numpy callable); mu is computed from it at the first solve"""
        cf = fem.Function(self.Vc); cf.interpolate(fun)
        self.w.x.array[self.mapc] = cf.x.array; self.w.x.scatter_forward()
        self.w0.x.array[:] = self.w.x.array

    def step(self):
        self.w0.x.array[:] = self.w.x.array
        self.pb.solve(); s = self.pb.solver; r = int(s.getConvergedReason()); n = int(s.getIterationNumber())
        self.t += float(self.dt.value); self.its.append(n)
        return r, n

    def c(self):
        return self.w.sub(0).collapse()

    def energy(self):
        c = ufl.split(self.w)[0]
        f = quartic(c) if self.potential == "quartic" else flory_huggins(c, self.chi)
        return comm.allreduce(fem.assemble_scalar(fem.form((f + 0.5 * self.eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(c))) * ufl.dx)), op=MPI.SUM)

    def mass(self):
        c = ufl.split(self.w)[0]
        return comm.allreduce(fem.assemble_scalar(fem.form(c * ufl.dx)), op=MPI.SUM)

    def mode(self, k, L=1.0):
        """amplitude of cos(k x) in c: 2/(L H) int c cos(k x)"""
        c = ufl.split(self.w)[0]; x = ufl.SpatialCoordinate(self.msh)
        area = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=self.msh))), op=MPI.SUM)
        return 2 / area * comm.allreduce(fem.assemble_scalar(fem.form(c * ufl.cos(k * x[0]) * ufl.dx)), op=MPI.SUM)

def growth_rate(k, eps, M=1.0, c0=0.0, fpp=None):
    """linear growth rate of a mode cos(k x) about c0: sigma = -M k^2 (f''(c0) + eps^2 k^2); f'' = 3 c0^2 - 1 (quartic)"""
    fpp = 3 * c0 ** 2 - 1 if fpp is None else fpp
    return -M * k ** 2 * (fpp + eps ** 2 * k ** 2)

def tanh_profile(x, eps):
    """1D equilibrium interface of the quartic potential: c = tanh(x / (sqrt(2) eps)); interfacial energy sigma = 2 sqrt(2) eps / 3"""
    return np.tanh(x / (np.sqrt(2) * eps))

def fh_binodal(chi, tol=1e-14):
    """symmetric Flory-Huggins (N1 = N2 = 1): coexisting phi from ln(phi/(1 - phi)) + chi (1 - 2 phi) = 0, phi < 1/2 (bisection)"""
    if chi <= 2:
        return 0.5, 0.5
    g = lambda p: np.log(p / (1 - p)) + chi * (1 - 2 * p)
    a, b = 1e-300, 0.5 - 1e-12
    for _ in range(200):
        m = 0.5 * (a + b)
        if g(a) * g(m) <= 0:
            b = m
        else:
            a = m
        if b - a < tol:
            break
    p = 0.5 * (a + b)
    return p, 1 - p

# ═══════════════════════════════════════════════ parallel performance
def wtime():
    comm.Barrier(); return MPI.Wtime()

def poisson(N, dim=3, deg=1):
    """-lap u = f on the unit square/cube, u = 0 on the boundary, u = exp(x + y) prod x_i (1 - x_i): returns (V, a, L, bcs, u_exact)"""
    msh = (mesh.create_unit_cube(comm, N, N, N, mesh.CellType.hexahedron) if dim == 3
           else mesh.create_unit_square(comm, N, N, mesh.CellType.quadrilateral))
    V = fem.functionspace(msh, ("Lagrange", deg)); u, v = ufl.TrialFunction(V), ufl.TestFunction(V); x = ufl.SpatialCoordinate(msh)
    ue = ufl.exp(x[0] + x[1])                                       # not an eigenfunction of the discrete operator (a sine product is,
    for i in range(dim):                                            # on a uniform tensor mesh: CG would then converge in one step)
        ue = ue * x[i] * (1 - x[i])
    f = -ufl.div(ufl.grad(ue))
    fdim = dim - 1; bnd = mesh.locate_entities_boundary(msh, fdim, lambda x: np.full(x.shape[1], True))
    bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(V, fdim, bnd), V)
    return V, ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx, f * v * ufl.dx, [bc], ue

def solve_poisson(N, method="gamg", dim=3, deg=1, rtol=1e-8):
    """times (max over ranks) of assembly, preconditioner setup and solve, iterations and the L2 error.
    method: "lu" (MUMPS), "gamg" (CG + algebraic multigrid), "jacobi" (CG + Jacobi), "mf" (matrix-free CG, no matrix, no preconditioner)"""
    collect()
    V, a, L, bcs, ue = poisson(N, dim, deg); msh = V.mesh; uh = fem.Function(V)
    out = dict(N=N, dofs=V.dofmap.index_map.size_global, ranks=comm.size, method=method)
    t0 = wtime(); b = fp.assemble_vector(fem.form(L)); fp.apply_lifting(b, [fem.form(a)], [bcs]); b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    fp.set_bc(b, bcs)
    ksp = PETSc.KSP().create(comm)
    if method == "mf":
        wf = fem.Function(V); actf = fem.form(ufl.action(a, wf)); yg = fp.create_vector(V)
        nown = V.dofmap.index_map.size_local * V.dofmap.index_map_bs
        bdofs = bcs[0].dof_indices()[0]; bown = bdofs[bdofs < nown]
        class Shell:
            """y = A x without a matrix: assemble the action of the bilinear form on x (into a ghosted work vector)"""
            def mult(self, mat, X, Y):
                wf.x.array[:nown] = X.array_r; wf.x.scatter_forward()
                with yg.localForm() as yl:
                    yl.set(0.0)
                fp.assemble_vector(yg, actf); yg.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
                y = Y.array; y[:] = yg.array_r[:nown]; y[bown] = X.array_r[bown]      # identity rows for the Dirichlet dofs
        A = PETSc.Mat().createPython([[nown, PETSc.DETERMINE], [nown, PETSc.DETERMINE]], Shell(), comm=comm); A.setUp()
        t1 = wtime(); out["assemble"] = t1 - t0; out["nnz"] = 0
        ksp.setOperators(A); ksp.setType("cg"); ksp.getPC().setType("none")
    else:
        A = fp.assemble_matrix(fem.form(a), bcs=bcs); A.assemble()
        t1 = wtime(); out["assemble"] = t1 - t0; out["nnz"] = A.getInfo(PETSc.Mat.InfoType.GLOBAL_SUM)["nz_used"]
        ksp.setOperators(A)
        if method == "lu":
            ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
        else:
            ksp.setType("cg"); ksp.getPC().setType("gamg" if method == "gamg" else "jacobi")
    ksp.setTolerances(rtol=rtol, atol=1e-50, max_it=20000)
    t2 = wtime(); ksp.setUp(); t3 = wtime(); out["setup"] = t3 - t2
    ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward(); t4 = wtime(); out["solve"] = t4 - t3
    out["its"] = ksp.getIterationNumber(); out["reason"] = ksp.getConvergedReason()
    out["error"] = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - ue) ** 2 * ufl.dx)), op=MPI.SUM))
    ksp.destroy(); A.destroy(); b.destroy()
    if method == "mf":
        yg.destroy()
    return out

def ghost_stats(msh):
    """per-rank owned and ghost cells and vertices, and the number of neighbouring ranks (gathered on every rank)"""
    tdim = msh.topology.dim; msh.topology.create_connectivity(0, tdim)
    ic, iv = msh.topology.index_map(tdim), msh.topology.index_map(0)
    loc = [ic.size_local, ic.num_ghosts, iv.size_local, iv.num_ghosts, len(set(iv.owners.tolist()))]
    return np.array(comm.allgather(loc))

class CahnHilliardLinear:
    """the stabilised scheme as a LINEAR problem: the matrix (c, mu) is constant, assembled and factorised ONCE; each step
    assembles the right-hand side and does two triangular solves. Same result as CahnHilliard(scheme="stab")."""
    count = 0
    def __init__(self, msh, eps=0.05, M=1.0, dt=1e-4, S=2.0, deg=1, reuse=True):
        collect(); CahnHilliardLinear.count += 1; k = CahnHilliardLinear.count
        self.msh, self.eps, self.reuse = msh, eps, reuse
        cell = msh.basix_cell(); P = basix.ufl.element("Lagrange", cell, deg)
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([P, P]))
        self.w, self.w0 = fem.Function(W), fem.Function(W)
        (c, mu), (q, v) = ufl.TrialFunctions(W), ufl.TestFunctions(W); c0 = ufl.split(self.w0)[0]
        a = (c / dt * q + M * ufl.dot(ufl.grad(mu), ufl.grad(q)) + mu * v - S * c * v - eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(v))) * ufl.dx
        L = (c0 / dt * q + (c0 ** 3 - c0 - S * c0) * v) * ufl.dx
        self.a, self.L = fem.form(a), fem.form(L)
        self.A = fp.assemble_matrix(self.a); self.A.assemble(); self.b = fp.create_vector(W)
        self.ksp = PETSc.KSP().create(comm); self.ksp.setOperators(self.A); self.ksp.setType("preonly")
        self.ksp.getPC().setType("lu"); self.ksp.getPC().setFactorSolverType("mumps"); self.ksp.setOptionsPrefix(f"chl{k}_")
        self.Vc, self.mapc = W.sub(0).collapse(); self.t = 0.0; self.dt = dt

    def set_c(self, fun):
        cf = fem.Function(self.Vc); cf.interpolate(fun); self.w.x.array[self.mapc] = cf.x.array; self.w.x.scatter_forward()

    def step(self):
        self.w0.x.array[:] = self.w.x.array
        if not self.reuse:                                              # re-assemble and re-factorise every step (what a Newton loop does)
            self.A.zeroEntries(); fp.assemble_matrix(self.A, self.a); self.A.assemble(); self.ksp.setOperators(self.A)
        with self.b.localForm() as bl:
            bl.set(0.0)
        fp.assemble_vector(self.b, self.L); self.b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        self.ksp.solve(self.b, self.w.x.petsc_vec); self.w.x.scatter_forward(); self.t += self.dt

    def energy(self):
        c = ufl.split(self.w)[0]
        return comm.allreduce(fem.assemble_scalar(fem.form((quartic(c) + 0.5 * self.eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(c))) * ufl.dx)), op=MPI.SUM)

## Q1 — Theory (15 pts)

**(a) (4)** Linearise $c_t=\nabla\cdot(M\nabla\mu)$, $\mu=f'(c)-\varepsilon^2\Delta c$ about a uniform $c_0$ and derive $\sigma(k)$. For $f=\frac14(c^2-1)^2$: which $c_0$ are unstable (spinodal), and what are $k_{\max}$ and $\sigma_{\max}$ at $c_0=0$?

**(b) (4)** Derive the amplification factors of backward Euler, convex splitting ($f'=c^3-c$: $c^3$ implicit, $-c$ explicit) and the stabilised scheme ($f'(c^n)+S(c-c^n)$) for one mode about $c_0=0$. Show that the last two are bounded for every $\Delta t$ and every $k$.

**(c) (4)** Convex splitting writes $f=f_c-f_e$ with $f_c,f_e$ convex, $f_c'$ implicit, $f_e'$ explicit. Show $E(c^{n+1})\le E(c^n)$ for every $\Delta t$ (use $f_c(b)-f_c(a)\le f_c'(b)(b-a)$ and $f_e(b)-f_e(a)\ge f_e'(a)(b-a)$).

**(d) (3)** For the symmetric Flory–Huggins free energy ($N_1=N_2=1$) derive the binodal condition and the spinodal $\phi(1-\phi)=1/(2\chi)$. Above which $\chi$ does the blend separate?

> *your answer*

## Q2 — Growth rates: time and space convergence (14 pts)

**(a) (8)** For the three schemes and $\Delta t=4\times10^{-4},2\times10^{-4},10^{-4},5\times10^{-5}$ (strip $128\times2$, $\varepsilon=0.05$, $k=4\pi$, up to $t=0.02$): $\sigma_h$, the relative error against $\sigma$, the observed order.

**(b) (6)** At $\Delta t=5\times10^{-5}$ compare $\sigma_h$ with each scheme's own discrete rate for 32, 64, 128 cells. Which part of the error is spatial?

In [ ]:
%%fenicsx -np 2
# TODO: Q2 — growth rates

> *your answer (Q2)*

## Q3 — The equilibrium interface: profile and interfacial energy (14 pts)

A 1D interface ($x\in[-1,1]$, a strip one cell high) started at twice its equilibrium width relaxes (convex splitting, $\Delta t=10^{-3}$, 400 steps) to $c=\tanh(x/(\sqrt2\varepsilon))$ with the interfacial energy $\gamma=\frac{2\sqrt2}{3}\varepsilon$.

**(a) (8)** $\varepsilon=0.05$: the energy per unit length and the maximum profile error for 50, 100, 200, 400 cells; observed order of $\gamma$.

**(b) (6)** Repeat with $\varepsilon=0.025$ on 200 and 400 cells. How many cells per interface width ($\sqrt2\varepsilon$) do you need for 1 % in $\gamma$? What does that imply for a 2D simulation of $64\times64$ cells at $\varepsilon=0.02$?

In [ ]:
%%fenicsx -np 2
# TODO: Q3 — the tanh interface

> *your answer (Q3)*

## Q4 — Flory–Huggins: binodal compositions and a failing Newton step (13 pts)

`potential="fh"`, the volume fraction $\phi\in(0,1)$, mean 0.5 + noise, $64\times64$, $\varepsilon=0.02$, convex splitting with $\Delta t=10^{-5}$ (100 steps), $10^{-4}$ (150), $10^{-3}$ (150).

**(a) (8)** For $\chi=2.5$ and $3.0$: the 5th and 95th percentiles of the nodal $\phi$ against the binodal; the mass drift.

**(b) (5)** A single step of $\Delta t=10^{-3}$ from the initial state with backward Euler and with convex splitting: SNES reason and iterations. Explain the failure.

In [ ]:
%%fenicsx -np 2
# TODO: Q4 — Flory-Huggins

> *your answer (Q4)*

## Q5 — Linear solvers: cost, iterations, memory, ghosts (15 pts)

3D Poisson, Q1 hexahedra, $N=12,16,24,32,40$ ($N^3$ cells), `-np 2`.

**(a) (9)** Setup + solve time and iterations for `lu`, `gamg`, `jacobi` (and `mf` up to $N=32$); fit the time exponents $t\propto n^p$ ($n$ unknowns) between the two largest sizes. Memory: nonzeros of the matrix.

**(b) (6)** The ghost vertices per rank and their fraction for $N=16,32$. How does the fraction scale with $N$ at fixed rank count, and with the rank count at fixed $N$ (lecture §C3)?

In [ ]:
%%fenicsx -np 2
# TODO: Q5 — solvers and ghosts

> *your answer (Q5)*

## Q6 — Accuracy per cost: Newton against a reused factorisation (14 pts)

**(a) (7)** Time per step on $64\times64$ and $128\times128$ for backward Euler (Newton), the stabilised scheme with Newton, re-assembled linear, and linear with one factorisation.

**(b) (7)** From Q2 the growth-rate error is about $c\,\Delta t$ with $c$ depending on the scheme. Which $\Delta t$ does each scheme need for a 1 % error, and what is the cost per unit time ($1/\Delta t\times$ time per step) on $128\times128$? Which scheme wins?

In [ ]:
%%fenicsx -np 2
# TODO: Q6 — time per step

> *your answer (Q6)*

## Q7 — Debugging an LLM-generated Cahn–Hilliard script (15 pts)

An LLM was asked for "a DOLFINx 0.11 script for spinodal decomposition with the Cahn–Hilliard equation ($\varepsilon=0.02$, $M=1$, unit square, mean $c=0$) with an unconditionally stable time stepping, up to $t=1$, that verifies itself". It returned the script below.

**(a) (3)** Run it as received with `-np 2`, then change the first line to `-np 1` and run it again. What does it claim, and what differs?

**(b) (7)** Find the defects. For each: what it does to the result, whether the script's own check can see it, and a test that exposes it.

**(c) (5)** Correct it (the module may be used) and verify it.

In [ ]:
%%fenicsx -np 2
# LLM-generated (as received): spinodal decomposition with the Cahn-Hilliard equation, Crank-Nicolson time stepping
import numpy as np, ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem
comm = MPI.COMM_WORLD
eps, M, dt, theta = 0.02, 1.0, 5e-3, 0.5                    # interface parameter, mobility, time step; theta = 0.5: unconditionally stable
msh = mesh.create_unit_square(comm, 64, 64, mesh.CellType.quadrilateral)
P1 = basix.ufl.element("Lagrange", msh.basix_cell(), 1)
W = fem.functionspace(msh, basix.ufl.mixed_element([P1, P1]))
w, w0 = fem.Function(W), fem.Function(W)
c, mu = ufl.split(w); c0, mu0 = ufl.split(w0); q, v = ufl.TestFunctions(W)
mu_mid = (1 - theta) * mu0 + theta * mu
F = ((c - c0) / dt * q + M * ufl.dot(ufl.grad(mu_mid), ufl.grad(q)) + (mu - c ** 3 + c) * v - eps * ufl.dot(ufl.grad(c), ufl.grad(v))) * ufl.dx
np.random.seed(0)
V0, dofs = W.sub(0).collapse()
c_init = fem.Function(V0)
c_init.x.array[:] = 0.05 * (2 * np.random.rand(c_init.x.array.size) - 1)   # small random perturbation around c = 0
w.x.array[dofs] = c_init.x.array
problem = NonlinearProblem(F, w, petsc_options_prefix="ch_", petsc_options={"snes_type": "newtonls", "ksp_type": "preonly", "pc_type": "lu",
                                                                             "pc_factor_mat_solver_type": "mumps"})
energy = lambda: comm.allreduce(fem.assemble_scalar(fem.form((0.25 * (c ** 2 - 1) ** 2 + 0.5 * eps * ufl.dot(ufl.grad(c), ufl.grad(c))) * ufl.dx)), op=MPI.SUM)
mass = lambda: comm.allreduce(fem.assemble_scalar(fem.form(c * ufl.dx)), op=MPI.SUM)
E0, m0 = energy(), mass()
for n in range(200):
    w0.x.array[:] = w.x.array
    problem.solve()
E1, m1 = energy(), mass()
print(f"energy {E0:.5f} -> {E1:.5f}, mass {m0:.6e} -> {m1:.6e}  ->  {'verified (energy decreases, mass conserved)' if E1 < E0 and abs(m1 - m0) < 1e-10 else 'CHECK'}")

In [ ]:
%%fenicsx -np 2
# TODO: Q7(c) — the corrected script and its verification

> *your list of defects and tests (Q7b), and your verification (Q7c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …